In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ==========================================
# 1. 加载数据
# ==========================================
# 加载中科大/数学真值 (前 10,000 个黎曼零点)
true_zeros = np.load('riemann_10k_true.npy')

# 加载您通过 pipeline_survey.py 跑出来的大规模原始相角或特征值
# 注意：请替换为您实际保存的文件名！
# 如果保存的是复数特征值 eigvals：
# raw_data = np.load('simulated_eigvals_10k.npy')
# raw_phases = np.angle(raw_data)
# 假设您保存的是已经转换好的相角 (-pi 到 pi)：
raw_phases = np.load('simulated_raw_phases.npy') # <-- 替换为您的文件

# ==========================================
# 2. 🌟 核心突破：负能量物理过滤器 🌟
# ==========================================
# 物理逻辑：过滤掉实轴上的平庸解 (相位≈0) 和代表“影子骨架”的负能量态 (相位<0)
# 仅仅保留纯净的正向拓扑演化相角！
pure_positive_phases = raw_phases[raw_phases > 1e-8]

print(f"原始数据点数: {len(raw_phases)}")
print(f"滤除负能量后剩余的正能量点数: {len(pure_positive_phases)}")

# ==========================================
# 3. 排序与相位解卷 (Phase Unwrapping)
# ==========================================
# 此时的排序和解卷将毫无杂质地积累正向行程
sorted_phases = np.sort(pure_positive_phases)
unwrapped_phases = np.unwrap(sorted_phases)

# 确保对标长度一致 (取两者中较小的值，通常是 10000 左右)
N_compare = min(len(unwrapped_phases), len(true_zeros))
X = unwrapped_phases[:N_compare]
Y_true = true_zeros[:N_compare]

# ==========================================
# 4. 全局线性缩放 (The Scale Problem)
# ==========================================
# 寻找非自治系统的天然比例尺 a 和 截距 b (E = a * Phi + b)
slope, intercept, r_value, p_value, std_err = linregress(X, Y_true)
Y_pred = slope * X + intercept

print(f"\n全局线性拟合结果:")
print(f"R^2 (相关系数平方): {r_value**2:.6f}")
print(f"比例尺 (Slope a): {slope:.6f}")
print(f"截距 (Intercept b): {intercept:.6f}")

# ==========================================
# 5. 残差计算
# ==========================================
# 计算绝对残差和相对残差 (%)
residuals = Y_pred - Y_true
mean_abs_err = np.mean(np.abs(residuals))
relative_errors = (np.abs(residuals) / Y_true) * 100

print(f"平均绝对误差: {mean_abs_err:.4f}")
print(f"最大相对误差: {np.max(relative_errors):.4f}%")

# ==========================================
# 6. 顶刊级数据可视化 (复刻 3-Panel 对比图)
# ==========================================
plt.rcParams['font.family'] = 'serif'
fig = plt.figure(figsize=(15, 10), dpi=150)

# 图 1：原始累积相位 vs 真实能量 (验证缩放问题)
ax1 = plt.subplot(2, 2, 1)
ax1.plot(Y_true, X, color='navy', alpha=0.8, label='Simulated (Raw Unwrapped Phase)')
ax1.set_xlabel('True Riemann Zeros ($E_n$)', fontweight='bold')
ax1.set_ylabel('Cumulative Phase $\Phi$', fontweight='bold')
ax1.set_title('Raw Phase vs True Energy (Pure Positive States)', fontweight='bold')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# 图 2：线性缩放后的完美对齐
ax2 = plt.subplot(2, 2, 2)
ax2.plot(Y_pred, Y_true, color='forestgreen', alpha=0.8, label='Calibrated Simulation')
ax2.plot([0, max(Y_true)], [0, max(Y_true)], color='red', linestyle='--', alpha=0.6, label='Ideal y=x')
ax2.set_xlabel('Calibrated Simulation ($E_{pred}$)', fontweight='bold')
ax2.set_ylabel('True Zeros ($E_n$)', fontweight='bold')
ax2.set_title(f'After Linear Scaling (R² = {r_value**2:.6f})', fontweight='bold')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

# 图 3：终极残差图 (见证 4% 误差是否被抹平！)
ax3 = plt.subplot(2, 1, 2)
ax3.plot(range(1, N_compare + 1), residuals, color='gray', alpha=0.7, linewidth=1.5)
ax3.axhline(0, color='black', linestyle='--', linewidth=1.5)
ax3.set_xlabel('Index $n$', fontweight='bold')
ax3.set_ylabel('Error ($E_{pred} - E_n$)', fontweight='bold')
ax3.set_title(f'Final Residual Error (Mean Err={mean_abs_err:.4f})', fontweight='bold')
ax3.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

<>:74: SyntaxWarning: invalid escape sequence '\P'
<>:74: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipykernel_4030/777758657.py:74: SyntaxWarning: invalid escape sequence '\P'
  ax1.set_ylabel('Cumulative Phase $\Phi$', fontweight='bold')
/tmp/ipykernel_4030/777758657.py:74: SyntaxWarning: invalid escape sequence '\P'
  ax1.set_ylabel('Cumulative Phase $\Phi$', fontweight='bold')


FileNotFoundError: [Errno 2] No such file or directory: 'simulated_raw_phases.npy'